<a href="https://colab.research.google.com/github/kursatkara/MAE_5020_Spring_2026/blob/master/05_08_exact_DMD_unknown_2D_field.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MAE 5020 Guided Example: Exact DMD on an Unknown 2D Wake-Like Dataset

In this notebook, you will apply **exact Dynamic Mode Decomposition (DMD)** to an **unknown time-resolved 2D field** stored in a `.npz` file.

Your goal is to extract as much information as possible about:
- steady or slowly varying content,
- dominant oscillatory dynamics,
- decay or growth rates,
- spatial mode shapes,
- reconstruction quality,
- and sensitivity to rank truncation in the presence of noise.

This dataset is designed to feel closer to CFD/PIV-style data than the first toy problem, while still remaining manageable for a first guided exact DMD exercise.


## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.dpi'] = 120


## 2. Load the unknown dataset

This notebook is designed to work with:

`mae5020_dmd_unknown_wake_case.npz`

In Google Colab, upload the file when prompted.


In [ ]:

filename = "mae5020_dmd_unknown_wake_case.npz"

if not os.path.exists(filename):
    try:
        from google.colab import files
        print(f"Please upload {filename}")
        uploaded = files.upload()
        if filename not in uploaded:
            raise FileNotFoundError(f"{filename} was not uploaded.")
    except Exception as e:
        raise FileNotFoundError(
            f"Could not find {filename}. Place it in the working directory or upload it in Colab."
        ) from e

data = np.load(filename)
X = data["X"]
t = data["t"]
x = data["x"]
y = data["y"]
dt = float(data["dt"])
nx = int(data["nx"])
ny = int(data["ny"])

print(f"X shape     : {X.shape}")
print(f"grid        : ny={ny}, nx={nx}")
print(f"snapshots   : {len(t)}")
print(f"dt          : {dt}")


## 3. Helper functions

In [ ]:

def vec_to_field(v, ny, nx):
    return np.real(v).reshape(ny, nx)

def plot_field(field, x, y, title="", ax=None, cmap="RdBu_r", vmin=None, vmax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 3))
    im = ax.imshow(
        np.real(field),
        origin="lower",
        aspect="auto",
        extent=[x[0], x[-1], y[0], y[-1]],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    return im

def exact_dmd(X, r, dt):
    X1 = X[:, :-1]
    X2 = X[:, 1:]

    U, s, Vh = np.linalg.svd(X1, full_matrices=False)
    V = Vh.T

    Ur = U[:, :r]
    Sr = np.diag(s[:r])
    Vr = V[:, :r]

    A_tilde = Ur.T @ X2 @ Vr @ np.linalg.inv(Sr)
    eigvals, W = np.linalg.eig(A_tilde)
    Phi = X2 @ Vr @ np.linalg.inv(Sr) @ W
    mu = np.log(eigvals) / dt

    return {"U": U, "s": s, "V": V, "eigvals": eigvals, "Phi": Phi, "mu": mu}

def dmd_reconstruction(Phi, mu, x0, t):
    b = np.linalg.lstsq(Phi, x0, rcond=None)[0]
    time_dynamics = np.zeros((len(b), len(t)), dtype=complex)
    for k, tk in enumerate(t):
        time_dynamics[:, k] = b * np.exp(mu * tk)
    X_rec = np.real(Phi @ time_dynamics)
    return X_rec, b


def fmt_complex(z, ndigits=6):
    return f"{z.real:.{ndigits}f}{z.imag:+.{ndigits}f}j"

def fmt_real_list(values, ndigits=4):
    return "[" + ", ".join(f"{float(v):.{ndigits}f}" for v in values) + "]"



## 4. First inspection

Before doing DMD, inspect the data.

### Questions
- Does the field appear mostly steady, oscillatory, transient, or drifting?
- Is there evidence of one dominant frequency?
- Do you see noise?


In [ ]:

snapshot_ids = [0, 10, 30, 70, 120]

fig, axes = plt.subplots(1, len(snapshot_ids), figsize=(16, 3), constrained_layout=True)
vmax = np.percentile(np.abs(X), 99)
vmin = -vmax

for ax, k in zip(axes, snapshot_ids):
    plot_field(vec_to_field(X[:, k], ny, nx), x, y, title=f"t = {t[k]:.2f}", ax=ax, vmin=vmin, vmax=vmax)

plt.show()


In [ ]:
from matplotlib import animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(6, 3))
vmax = np.percentile(np.abs(X), 99)
vmin = -vmax

# Initial plot
field0 = vec_to_field(X[:, 0], ny, nx)
im = ax.imshow(
    field0,
    origin="lower",
    aspect="auto",
    extent=[x[0], x[-1], y[0], y[-1]],
    cmap="RdBu_r",
    vmin=vmin,
    vmax=vmax,
)
ax.set_xlabel("x")
ax.set_ylabel("y")
title = ax.set_title(f"t = {t[0]:.2f}")

def update(frame):
    field = vec_to_field(X[:, frame], ny, nx)
    im.set_array(field)
    title.set_text(f"t = {t[frame]:.2f}")
    return im, title

anim = animation.FuncAnimation(fig, update, frames=len(t), interval=50, blit=True)
plt.close(fig) # Prevent duplicate static plot

# Display the animation
HTML(anim.to_jshtml())

In [ ]:

Xgrid = X.reshape(ny, nx, len(t))

probe_locations = [(ny // 2, 20), (ny // 2 + 6, 30), (ny // 2 - 6, 45)]
plt.figure(figsize=(8, 4))
for iy, ix in probe_locations:
    plt.plot(t, Xgrid[iy, ix, :], label=f"(x={x[ix]:.2f}, y={y[iy]:.2f})")
plt.xlabel("Time")
plt.ylabel("Signal")
plt.title("Signals at selected spatial probe locations")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



## 5. Build the snapshot matrices

$$
X_1 =
\begin{bmatrix}
x_1 & x_2 & \cdots & x_{m-1}
\end{bmatrix},
\qquad
X_2 =
\begin{bmatrix}
x_2 & x_3 & \cdots & x_m
\end{bmatrix}.
$$

Exact DMD seeks a best-fit linear map such that

$$
x_{k+1} \approx A x_k.
$$


In [ ]:
X1 = X[:, :-1]
X2 = X[:, 1:]
print(X1.shape, X2.shape)


## 6. Singular values and rank selection

The singular values of $X_1$ help us estimate how many dominant directions are present before noise begins to matter.

A reasonable first try for this dataset is **$r=4$**:
- one nearly steady component,
- one oscillatory conjugate pair,
- one weaker real transient.


In [ ]:

U, s, Vh = np.linalg.svd(X1, full_matrices=False)
energy = s**2
cum_energy = np.cumsum(energy) / np.sum(energy)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].semilogy(s[:20], "o-")
ax[0].set_xlabel("Index")
ax[0].set_ylabel("Singular value")
ax[0].set_title("Leading singular values")
ax[0].grid(alpha=0.3)

ax[1].plot(cum_energy[:20], "o-")
ax[1].set_xlabel("Index")
ax[1].set_ylabel("Cumulative energy")
ax[1].set_title("Cumulative energy")
ax[1].grid(alpha=0.3)
plt.show()

for j in range(10):
    print(f"Mode {j+1:2d}: singular value = {s[j]:10.4f}, cumulative energy = {cum_energy[j]:.6f}")


In [ ]:
r = 4


## 7. Apply exact DMD

We compute the exact DMD modes and the continuous-time eigenvalues

$$
\mu_j = \frac{\log(\lambda_j)}{\Delta t}.
$$

Then:
- $\mathrm{Re}(\mu_j)$ gives growth or decay,
- $\mathrm{Im}(\mu_j)$ gives angular frequency,
- and frequency in Hz is $f_j = \mathrm{Im}(\mu_j)/(2\pi)$.


In [ ]:
dmd = exact_dmd(X, r=r, dt=dt)
eigvals = dmd["eigvals"]
Phi = dmd["Phi"]
mu = dmd["mu"]

print("  j         lambda                     growth rate      frequency [Hz]")
for j in range(r):
    lam_str = fmt_complex(eigvals[j], ndigits=6)
    growth = float(np.real(mu[j]))
    freq = float(np.imag(mu[j]) / (2*np.pi))
    print(f"{j:3d}   {lam_str:>25}   {growth:14.6f}   {freq:14.6f}")


In [ ]:

theta = np.linspace(0, 2*np.pi, 400)

plt.figure(figsize=(5, 5))
plt.plot(np.cos(theta), np.sin(theta), "k--", label="Unit circle")
plt.scatter(np.real(eigvals), np.imag(eigvals), s=80)
for j, lam in enumerate(eigvals):
    plt.text(np.real(lam) + 0.01, np.imag(lam) + 0.01, str(j), fontsize=9)
plt.xlabel("Real part")
plt.ylabel("Imaginary part")
plt.title(f"DMD eigenvalues (r = {r})")
plt.axis("equal")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 8. Visualize the DMD modes

In [ ]:

fig, axes = plt.subplots(2, r, figsize=(3.5 * r, 6), constrained_layout=True)

for j in range(r):
    plot_field(vec_to_field(np.real(Phi[:, j]), ny, nx), x, y, title=f"Mode {j}: real part", ax=axes[0, j])
    plot_field(vec_to_field(np.imag(Phi[:, j]), ny, nx), x, y, title=f"Mode {j}: imag part", ax=axes[1, j])

plt.show()



## 9. Reconstruction

We reconstruct the data as

$$
x(t_k) \approx \sum_{j=1}^r b_j \phi_j e^{\mu_j t_k}.
$$


In [ ]:

X_rec, b = dmd_reconstruction(Phi, mu, X[:, 0], t)
rel_err = np.linalg.norm(X - X_rec) / np.linalg.norm(X)
print(f"Relative reconstruction error = {rel_err:.6e}")
print("Modal amplitudes:")
print(b)


In [ ]:

compare_ids = [0, 20, 60, 120]

fig, axes = plt.subplots(len(compare_ids), 3, figsize=(10, 2.8 * len(compare_ids)), constrained_layout=True)
vmax = np.percentile(np.abs(X), 99)
vmin = -vmax

for i, k in enumerate(compare_ids):
    plot_field(vec_to_field(X[:, k], ny, nx), x, y, title=f"True snapshot, t={t[k]:.2f}", ax=axes[i, 0], vmin=vmin, vmax=vmax)
    plot_field(vec_to_field(X_rec[:, k], ny, nx), x, y, title=f"DMD reconstruction, t={t[k]:.2f}", ax=axes[i, 1], vmin=vmin, vmax=vmax)
    plot_field(vec_to_field(X[:, k] - X_rec[:, k], ny, nx), x, y, title=f"Error, t={t[k]:.2f}", ax=axes[i, 2], vmin=vmin, vmax=vmax)
plt.show()



## 10. Rank sensitivity

Because the data are noisy and not perfectly low rank, test several values of $r$.


In [ ]:
candidate_ranks = [2, 3, 4, 5, 6, 8]

for rr in candidate_ranks:
    out = exact_dmd(X, r=rr, dt=dt)
    Xr, _ = dmd_reconstruction(out["Phi"], out["mu"], X[:, 0], t)
    err = float(np.linalg.norm(X - Xr) / np.linalg.norm(X))
    freqs_rr = np.imag(out["mu"]) / (2*np.pi)

    pos_freqs = sorted(float(f) for f in freqs_rr if f > 1e-4)
    print(f"r = {rr:2d} | relative error = {err:10.6e} | positive frequencies = {fmt_real_list(pos_freqs, ndigits=4)}")



## 11. Guided interpretation

Use your results to answer:

1. Is there a strong mean or slowly varying mode?
2. Is there a dominant oscillatory conjugate pair? What is its frequency?
3. Is there evidence of a weaker decaying real mode?
4. Which rank seems most physically meaningful?
5. Which features appear robust, and which appear noise-sensitive?
